# 02b -- Preprocessing Helpers

This notebook is the preprocessing-only replica of Mohsen's helper workflow.

It preserves the preprocessing outputs that lead into feature extraction:

- minimum-duration filtering for `ACC_Z`
- retained/removed record summaries
- common-event discovery for one representative deck example
- multichannel waveform-matrix loading
- zero-phase `0.5-20 Hz` band-pass preparation

FDD, peak picking, mode-shape tables, mode-shape plots, and singular-value spectra remain in `03_feature_extraction.ipynb`.


In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

import matplotlib.pyplot as plt
import pandas as pd

from aquinas_toolkit import AquinasReader
from aquinas_toolkit.preprocessing import (
    bandpass_filter_waveform_matrix,
    find_common_sensor_events,
    load_common_event_waveform_matrix,
    summarize_min_duration_filter,
)

MIN_DURATION_SECONDS = 10.0
DATASET_ROOT = Path("../AQUINAS_DATASET")
SAMPLE_RATE_HZ = 100.0
LOW_HZ = 0.5
HIGH_HZ = 20.0
EXAMPLE_DECK = "OLD"


In [ ]:
set_summaries = []

for set_dir in sorted(DATASET_ROOT.glob("AQUINAS_SET*")):
    reader = AquinasReader(set_dir)
    sensor_summary = summarize_min_duration_filter(
        reader,
        min_duration_seconds=MIN_DURATION_SECONDS,
        quantity="ACC",
        axis="Z",
    )

    set_summaries.append(
        {
            "Dataset": reader.set_name,
            "Sensors": len(sensor_summary),
            "Total records": int(sensor_summary["record_count"].sum()),
            "Kept": int(sensor_summary["kept_count"].sum()),
            "Removed": int(sensor_summary["removed_count"].sum()),
            "Removed %": sensor_summary["removed_count"].sum() / sensor_summary["record_count"].sum(),
        }
    )

    display(
        sensor_summary[
            [
                "sensor_name",
                "deck",
                "span",
                "side",
                "location",
                "record_count",
                "kept_count",
                "removed_count",
                "kept_fraction",
            ]
        ]
        .rename(
            columns={
                "sensor_name": "Sensor",
                "deck": "Deck",
                "span": "Span",
                "side": "Side",
                "location": "Location",
                "record_count": "Total records",
                "kept_count": "Kept",
                "removed_count": "Removed",
                "kept_fraction": "Kept %",
            }
        )
        .style.hide(axis="index")
        .format({"Total records": "{:,.0f}", "Kept": "{:,.0f}", "Removed": "{:,.0f}", "Kept %": "{:.1%}"})
        .set_caption(
            f"{reader.set_name} — ACC_Z records kept after minimum duration filter ({MIN_DURATION_SECONDS:.0f} s)"
        )
    )

display(
    pd.DataFrame(set_summaries)
    .style.hide(axis="index")
    .format({"Sensors": "{:,.0f}", "Total records": "{:,.0f}", "Kept": "{:,.0f}", "Removed": "{:,.0f}", "Removed %": "{:.1%}"})
    .set_caption(
        f"ACC_Z minimum duration filter summary across all datasets ({MIN_DURATION_SECONDS:.0f} s threshold)"
    )
)


## Zero-phase band-pass preparation before FDD

This section stops exactly at the handoff point where `03_feature_extraction.ipynb` takes over.


In [ ]:
example_set_dir = sorted(DATASET_ROOT.glob("AQUINAS_SET*"))[0]
reader = AquinasReader(example_set_dir)
common_events = find_common_sensor_events(
    reader,
    min_duration_seconds=MIN_DURATION_SECONDS,
    quantity="ACC",
    axis="Z",
    deck=EXAMPLE_DECK,
)

display(Markdown(f"### {reader.set_name} | {EXAMPLE_DECK}"))
display(
    common_events[["Start_Time", "End_Time", "sensor_count", "channel_count"]]
    .rename(
        columns={
            "Start_Time": "Start time",
            "End_Time": "End time",
            "sensor_count": "Sensor count",
            "channel_count": "Channel count",
        }
    )
    .head(10)
    .style.hide(axis="index")
    .format({"Sensor count": "{:,.0f}", "Channel count": "{:,.0f}"})
    .set_caption(f"{reader.set_name} | {EXAMPLE_DECK} — first common ACC_Z events before filtering")
)


In [ ]:
event = common_events.iloc[0]

waveform_matrix = load_common_event_waveform_matrix(
    reader,
    start_time=event["Start_Time"],
    end_time=event["End_Time"],
    sensor_names=event["sensor_names"],
)
filtered_matrix = bandpass_filter_waveform_matrix(
    waveform_matrix,
    sampling_rate_hz=SAMPLE_RATE_HZ,
    low_hz=LOW_HZ,
    high_hz=HIGH_HZ,
)

display(
    waveform_matrix.head()
    .style.hide(axis="index")
    .set_caption(f"{reader.set_name} | {EXAMPLE_DECK} — raw waveform matrix preview")
)
display(
    filtered_matrix.head()
    .style.hide(axis="index")
    .set_caption(f"{reader.set_name} | {EXAMPLE_DECK} — zero-phase band-pass filtered waveform matrix preview")
)


In [ ]:
channels = [column for column in waveform_matrix.columns if column != "timestamp"]
plot_channels = channels[: min(3, len(channels))]

fig, axes = plt.subplots(len(plot_channels), 1, figsize=(10, 2.8 * len(plot_channels)), sharex=True)
if len(plot_channels) == 1:
    axes = [axes]

for axis, channel in zip(axes, plot_channels, strict=True):
    axis.plot(waveform_matrix.index, waveform_matrix[channel], label="Raw", alpha=0.8)
    axis.plot(filtered_matrix.index, filtered_matrix[channel], label="Filtered", alpha=0.8)
    axis.set_title(channel)
    axis.grid(alpha=0.25)
    axis.legend()

axes[-1].set_xlabel("Sample index")
fig.suptitle(f"{reader.set_name} | {EXAMPLE_DECK} — raw vs zero-phase band-pass filtered ACC_Z event", y=1.02)
fig.tight_layout()
plt.show()
